# Capítulo 7 — Funciones, manejo de errores y módulos

**Serie Ciencia de Datos · Tomo 1 — Introducción al Data Science y Programación con Python**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes. Las celdas que usan `input()` esperan que usted escriba un valor.

## Código del capítulo

### 7.1 Crear sus propias funciones

In [ ]:
def limpiar_ciudad(ciudad):
    """Devuelve la ciudad sin espacios de más, en minúsculas y sin tildes."""
    ciudad = ciudad.strip().lower()
    for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
        ciudad = ciudad.replace(con_tilde, sin_tilde)
    return ciudad


# Una vez creada, la función se usa como cualquier otra.
print(limpiar_ciudad("  BOGOTÁ "))
print(limpiar_ciudad("Medellín"))

*Salida en el libro:*

```text
bogota
medellin
```

#### Varios parámetros y valores por defecto

In [ ]:
def calcular_total(precio, cantidad, descuento=0.0):
    """Devuelve el total de una venta.

    precio: precio de una unidad, en pesos.
    cantidad: número de unidades.
    descuento: porcentaje de descuento entre 0 y 1 (por defecto, ninguno).
    """
    return precio * cantidad * (1 - descuento)


print(calcular_total(45000, 3))           # sin descuento
print(calcular_total(45000, 3, 0.10))     # con descuento del 10 %
print(calcular_total(precio=45000, cantidad=3, descuento=0.10))

*Salida en el libro:*

```text
135000.0
121500.0
121500.0
```

#### Cumplir una promesa: el índice de Jaccard como función

In [ ]:
def indice_jaccard(conjunto_a, conjunto_b):
    """Devuelve el índice de Jaccard entre dos conjuntos (de 0 a 1).

    Si los dos conjuntos están vacíos, devuelve 0.0.
    """
    union = conjunto_a | conjunto_b
    if len(union) == 0:
        return 0.0
    return len(conjunto_a & conjunto_b) / len(union)


compras_ana = {"camisa", "jean", "medias", "gorra"}
compras_luis = {"camisa", "jean", "chaqueta"}

print(f"{indice_jaccard(compras_ana, compras_luis):.2f}")
print(f"{indice_jaccard(set(), set()):.2f}")

*Salida en el libro:*

```text
0.40
0.00
```

#### [Ex-L1-C7-01] — Funciones de limpieza

In [ ]:
pedidos_crudos = [("$ 45.000", 3), ("$ 120.000", 0), ("$ 1.250.000", 1),
                  ("$ 35.000", 250), ("$ 89.900", 2)]

### 7.2 Funciones que reciben funciones: ordenar con criterio

In [ ]:
productos = [("camisa", 45000), ("chaqueta", 230000), ("medias", 15000)]


def obtener_precio(producto):
    """Devuelve el precio de una tupla (nombre, precio)."""
    return producto[1]


# key=obtener_precio: ordena según lo que devuelve esa función.
print(sorted(productos, key=obtener_precio))
print(max(productos, key=obtener_precio))

*Salida en el libro:*

```text
[('medias', 15000), ('camisa', 45000), ('chaqueta', 230000)]
('chaqueta', 230000)
```

In [ ]:
productos = [("camisa", 45000), ("chaqueta", 230000), ("medias", 15000)]

# lambda producto: producto[1] es una función que recibe un producto
# y devuelve su precio. Hace lo mismo que obtener_precio.
print(sorted(productos, key=lambda producto: producto[1], reverse=True))

*Salida en el libro:*

```text
[('chaqueta', 230000), ('camisa', 45000), ('medias', 15000)]
```

In [ ]:
total_por_categoria = {"ropa": 455000, "hogar": 187500, "electronica": 390000}

# .items() entrega pares (categoría, total); se ordena por el total.
ranking = sorted(total_por_categoria.items(),
                 key=lambda par: par[1], reverse=True)
for posicion, (categoria, total) in enumerate(ranking, start=1):
    print(f"{posicion}. {categoria}: {total} pesos")

*Salida en el libro:*

```text
1. ropa: 455000 pesos
2. electronica: 390000 pesos
3. hogar: 187500 pesos
```

#### [Ex-L1-C7-02] — Los productos estrella

In [ ]:
unidades_vendidas = {"camisa": 120, "jean": 85, "chaqueta": 30,
                     "medias": 210, "gorra": 45, "sarten": 12}
precios = {"camisa": 45000, "jean": 120000, "chaqueta": 230000,
           "medias": 15000, "gorra": 35000, "sarten": 89000}

### 7.3 Lo que pasa dentro de una función se queda dentro

In [ ]:
def calcular_iva(valor):
    """Devuelve el IVA del 19 % de un valor."""
    iva_calculado = valor * 0.19
    return iva_calculado


resultado = calcular_iva(100000)
print(resultado)

# iva_calculado solo existía dentro de la función:
print("iva_calculado" in dir())

*Salida en el libro:*

```text
19000.0
False
```

### 7.4 Prepararse para los errores: try y except

In [ ]:
precios_texto = ["45000", "cuarenta mil", "120000"]

total = 0
for precio in precios_texto:
    try:
        total += int(precio)
    except ValueError:
        print(f"Se omitió un precio que no es un número: '{precio}'")

print(f"Total: {total} pesos")

*Salida en el libro:*

```text
Se omitió un precio que no es un número: 'cuarenta mil'
Total: 165000 pesos
```

#### Atrape solo los errores que espera

In [ ]:
def precio_por_unidad(total_texto, cantidad):
    """Divide el total de un pedido entre la cantidad de unidades."""
    try:
        total = int(total_texto)
        resultado = total / cantidad
    except ValueError:
        print(f"El total '{total_texto}' no es un número.")
        return None
    except ZeroDivisionError:
        print("La cantidad es cero: no se puede calcular el precio.")
        return None
    else:
        # else se ejecuta solo si no hubo ningún error.
        return resultado
    finally:
        # finally se ejecuta siempre, con error o sin él.
        print("Revisión del pedido terminada.")


print(precio_por_unidad("90000", 2))
print(precio_por_unidad("noventa mil", 2))
print(precio_por_unidad("90000", 0))

*Salida en el libro:*

```text
Revisión del pedido terminada.
45000.0
El total 'noventa mil' no es un número.
Revisión del pedido terminada.
None
La cantidad es cero: no se puede calcular el precio.
Revisión del pedido terminada.
None
```

#### Avisar de un error: raise

In [ ]:
def promedio(valores):
    """Devuelve el promedio de una lista de números.

    Produce un ValueError si la lista está vacía.
    """
    if len(valores) == 0:
        raise ValueError("No se puede calcular el promedio de una lista "
                         "vacía.")
    return sum(valores) / len(valores)


print(promedio([45000, 120000, 35000]))

try:
    promedio([])
except ValueError as error:
    print(f"Error: {error}")

*Salida en el libro:*

```text
66666.66666666667
Error: No se puede calcular el promedio de una lista vacía.
```

#### [Ex-L1-C7-03] — Su propia función de resumen

In [ ]:
ventas_lunes = ["45000", "120000", "35000", "89900"]
ventas_martes = ["45000", "N/A", "230000", "", "67000"]
ventas_miercoles = ["sin dato", "N/A"]

### 7.5 Guardar sus funciones en un archivo: los módulos

In [ ]:
from math import isclose

print(isclose(0.1 + 0.2, 0.3))

*Salida en el libro:*

```text
True
```

#### Crear su propio módulo en Colab

In [ ]:
%%writefile utilidades_tienda.py
"""Funciones de limpieza para los datos de la tienda."""


def limpiar_ciudad(ciudad):
    """Devuelve la ciudad sin espacios de más, en minúsculas y sin tildes."""
    ciudad = ciudad.strip().lower()
    for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
        ciudad = ciudad.replace(con_tilde, sin_tilde)
    return ciudad

*Salida en el libro:*

```text
Writing utilidades_tienda.py
```

In [ ]:
import utilidades_tienda

print(utilidades_tienda.limpiar_ciudad(" CÚCUTA "))

*Salida en el libro:*

```text
cucuta
```

#### [Ex-L1-C7-04] — Su módulo de limpieza

In [ ]:
pedidos = [
    {"ciudad": " Bogotá", "precio": "$ 45.000", "cantidad": 3},
    {"ciudad": "CALI ", "precio": "sin precio", "cantidad": 1},
    {"ciudad": "medellín", "precio": "$ 120.000", "cantidad": 0},
    {"ciudad": "Cali", "precio": "$ 89.900", "cantidad": 2},
]

## Soluciones de los ejercicios

### [Ex-L1-C7-01] — Funciones de limpieza

In [ ]:
pedidos_crudos = [("$ 45.000", 3), ("$ 120.000", 0), ("$ 1.250.000", 1),
                  ("$ 35.000", 250), ("$ 89.900", 2)]

In [ ]:
def limpiar_precio(texto):
    """Convierte un precio en formato colombiano ('$ 1.250.000') en entero.

    Quita el signo de pesos, los puntos de miles y los espacios.
    """
    texto_limpio = texto.replace("$", "").replace(".", "").strip()
    return int(texto_limpio)


def es_pedido_valido(precio, cantidad):
    """Devuelve True si el precio es positivo y la cantidad va de 1 a 100."""
    return precio > 0 and 1 <= cantidad <= 100


pedidos_crudos = [("$ 45.000", 3), ("$ 120.000", 0), ("$ 1.250.000", 1),
                  ("$ 35.000", 250), ("$ 89.900", 2)]

total_validos = 0
for precio_texto, cantidad in pedidos_crudos:
    precio = limpiar_precio(precio_texto)
    if es_pedido_valido(precio, cantidad):
        total_validos += precio * cantidad
        print(f"Válido:   {precio} x {cantidad}")
    else:
        print(f"Inválido: {precio} x {cantidad}")

print(f"Total de los pedidos válidos: {total_validos} pesos")

*Salida en el libro:*

```text
Válido:   45000 x 3
Inválido: 120000 x 0
Válido:   1250000 x 1
Inválido: 35000 x 250
Válido:   89900 x 2
Total de los pedidos válidos: 1564800 pesos
```

### [Ex-L1-C7-02] — Los productos estrella

In [ ]:
unidades_vendidas = {"camisa": 120, "jean": 85, "chaqueta": 30,
                     "medias": 210, "gorra": 45, "sarten": 12}
precios = {"camisa": 45000, "jean": 120000, "chaqueta": 230000,
           "medias": 15000, "gorra": 35000, "sarten": 89000}

In [ ]:
unidades_vendidas = {"camisa": 120, "jean": 85, "chaqueta": 30,
                     "medias": 210, "gorra": 45, "sarten": 12}
precios = {"camisa": 45000, "jean": 120000, "chaqueta": 230000,
           "medias": 15000, "gorra": 35000, "sarten": 89000}

# --- 1. Los tres productos con más unidades vendidas ---
top_unidades = sorted(unidades_vendidas.items(),
                      key=lambda par: par[1], reverse=True)[:3]
print("Más unidades vendidas:")
for producto, unidades in top_unidades:
    print(f"  {producto}: {unidades} unidades")

# --- 2. Dinero que trajo cada producto ---
ingresos = {producto: unidades * precios[producto]
            for producto, unidades in unidades_vendidas.items()}

# --- 3. Los tres productos que más dinero trajeron ---
top_ingresos = sorted(ingresos.items(),
                      key=lambda par: par[1], reverse=True)[:3]
print("Más dinero:")
for producto, ingreso in top_ingresos:
    print(f"  {producto}: {ingreso} pesos")

# --- 4. El producto más caro ---
producto_mas_caro = max(precios, key=lambda producto: precios[producto])
print(f"Producto más caro: {producto_mas_caro}")

*Salida en el libro:*

```text
Más unidades vendidas:
  medias: 210 unidades
  camisa: 120 unidades
  jean: 85 unidades
Más dinero:
  jean: 10200000 pesos
  chaqueta: 6900000 pesos
  camisa: 5400000 pesos
Producto más caro: chaqueta
```

### [Ex-L1-C7-03] — Su propia función de resumen

In [ ]:
ventas_lunes = ["45000", "120000", "35000", "89900"]
ventas_martes = ["45000", "N/A", "230000", "", "67000"]
ventas_miercoles = ["sin dato", "N/A"]

In [ ]:
def resumir_ventas(ventas_texto):
    """Resume una lista de ventas escritas como texto.

    Omite los valores que no se pueden convertir en número y los cuenta.

    Recibe: una lista de textos, por ejemplo ["45000", "N/A"].
    Devuelve: un diccionario con numero_ventas, total, promedio, maxima,
    minima y valores_invalidos.
    Produce ValueError si no queda ninguna venta válida.
    """
    ventas_validas = []
    valores_invalidos = 0
    for venta_texto in ventas_texto:
        try:
            ventas_validas.append(int(venta_texto))
        except ValueError:
            valores_invalidos += 1

    if len(ventas_validas) == 0:
        raise ValueError("No hay ninguna venta válida para resumir.")

    total = sum(ventas_validas)
    return {
        "numero_ventas": len(ventas_validas),
        "total": total,
        "promedio": total / len(ventas_validas),
        "maxima": max(ventas_validas),
        "minima": min(ventas_validas),
        "valores_invalidos": valores_invalidos,
    }


ventas_lunes = ["45000", "120000", "35000", "89900"]
ventas_martes = ["45000", "N/A", "230000", "", "67000"]
ventas_miercoles = ["sin dato", "N/A"]

# Mostramos cada resumen con un dato por línea, para que se lea mejor.
print("Lunes:")
for nombre, valor in resumir_ventas(ventas_lunes).items():
    print(f"  {nombre}: {valor}")

print("Martes:")
for nombre, valor in resumir_ventas(ventas_martes).items():
    print(f"  {nombre}: {valor}")

try:
    resumen_miercoles = resumir_ventas(ventas_miercoles)
except ValueError as error:
    print(f"Miércoles: {error}")

*Salida en el libro:*

```text
Lunes:
  numero_ventas: 4
  total: 289900
  promedio: 72475.0
  maxima: 120000
  minima: 35000
  valores_invalidos: 0
Martes:
  numero_ventas: 3
  total: 342000
  promedio: 114000.0
  maxima: 230000
  minima: 45000
  valores_invalidos: 2
Miércoles: No hay ninguna venta válida para resumir.
```

### [Ex-L1-C7-04] — Su módulo de limpieza

In [ ]:
pedidos = [
    {"ciudad": " Bogotá", "precio": "$ 45.000", "cantidad": 3},
    {"ciudad": "CALI ", "precio": "sin precio", "cantidad": 1},
    {"ciudad": "medellín", "precio": "$ 120.000", "cantidad": 0},
    {"ciudad": "Cali", "precio": "$ 89.900", "cantidad": 2},
]

In [ ]:
%%writefile limpieza_pedidos.py
"""Funciones para limpiar y revisar los pedidos de la tienda."""


def limpiar_ciudad(ciudad):
    """Devuelve la ciudad sin espacios de más, en minúsculas y sin tildes."""
    ciudad = ciudad.strip().lower()
    for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
        ciudad = ciudad.replace(con_tilde, sin_tilde)
    return ciudad


def limpiar_precio(texto):
    """Convierte un precio en formato colombiano ('$ 1.250.000') en entero."""
    texto_limpio = texto.replace("$", "").replace(".", "").strip()
    return int(texto_limpio)


def revisar_pedido(pedido):
    """Devuelve una copia limpia del pedido con la clave 'valido'.

    Recibe un diccionario con 'ciudad', 'precio' (texto) y 'cantidad'.
    Si el precio no se puede convertir, 'precio' queda en None y el
    pedido no es válido.
    """
    try:
        precio = limpiar_precio(pedido["precio"])
    except ValueError:
        precio = None

    cantidad = pedido["cantidad"]
    valido = (precio is not None and precio > 0
              and 1 <= cantidad <= 100)

    return {
        "ciudad": limpiar_ciudad(pedido["ciudad"]),
        "precio": precio,
        "cantidad": cantidad,
        "valido": valido,
    }

*Salida en el libro:*

```text
Writing limpieza_pedidos.py
```

In [ ]:
import limpieza_pedidos

pedidos = [
    {"ciudad": " Bogotá", "precio": "$ 45.000", "cantidad": 3},
    {"ciudad": "CALI ", "precio": "sin precio", "cantidad": 1},
    {"ciudad": "medellín", "precio": "$ 120.000", "cantidad": 0},
    {"ciudad": "Cali", "precio": "$ 89.900", "cantidad": 2},
]

total_validos = 0
for pedido in pedidos:
    revisado = limpieza_pedidos.revisar_pedido(pedido)
    print(revisado)
    if revisado["valido"]:
        total_validos += revisado["precio"] * revisado["cantidad"]

print(f"Total de los pedidos válidos: {total_validos} pesos")

*Salida en el libro:*

```text
{'ciudad': 'bogota', 'precio': 45000, 'cantidad': 3, 'valido': True}
{'ciudad': 'cali', 'precio': None, 'cantidad': 1, 'valido': False}
{'ciudad': 'medellin', 'precio': 120000, 'cantidad': 0, 'valido': False}
{'ciudad': 'cali', 'precio': 89900, 'cantidad': 2, 'valido': True}
Total de los pedidos válidos: 314800 pesos
```